> **Prerequisite:** Run earlier notebooks in numeric order in the **same kernel session**.

# 02 — Panel plotting functions

Style constants and drawing routines for panels a–d.

In [1]:
"""Panel plotting for salience topology mediation figure (HMM states 3–6)."""
from __future__ import annotations

import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats as scipy_stats
from statsmodels.api import OLS, add_constant

import matplotlib as mpl
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.legend_handler import HandlerBase
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

def _resolve_package_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "analysis_input.csv").is_file():
            return candidate
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
        nested = candidate / "mediate_moderate_lesion_analysis"
        if (nested / "data").is_dir() and (nested / "notebooks").is_dir():
            return nested
    if start.name == "notebooks" and (start.parent / "data").is_dir():
        return start.parent
    raise FileNotFoundError(
        "Could not locate mediate_moderate_lesion_analysis package root "
        f"(started search from: {start})"
    )


NOTEBOOK_DIR = Path.cwd()
MEDIATE_MODERATE_LESION_ROOT = _resolve_package_root()
DATA_DIR = MEDIATE_MODERATE_LESION_ROOT / "data"
FIGURES_DIR = MEDIATE_MODERATE_LESION_ROOT / "figures"
OUT_DIR = FIGURES_DIR
OUT_BASENAME = "salience_topology_mediation"

def _in_notebook() -> bool:
    try:
        return get_ipython() is not None
    except NameError:
        return False


def _enable_inline_backend() -> None:
    if not _in_notebook():
        return
    try:
        get_ipython().run_line_magic("matplotlib", "inline")
    except Exception:
        pass


def _show_figure(fig: plt.Figure) -> None:
    """Notebook preview with white background so updates are visible."""
    if not _in_notebook():
        plt.show()
        return
    _enable_inline_backend()
    from IPython.display import display

    saved_fc = fig.patch.get_facecolor()
    saved_alpha = fig.patch.get_alpha()
    fig.patch.set_facecolor("white")
    fig.patch.set_alpha(1.0)
    display(fig)
    fig.patch.set_facecolor(saved_fc)
    fig.patch.set_alpha(saved_alpha)


# Text/layout-only knobs (data graphics unchanged)
LAYOUT = {
    "suptitle_y": 0.99,
    "adj_top": 0.94,
    "adj_bottom": 0.155,
    "adj_left": 0.08,
    "adj_right": 0.98,
    "wspace": 0.365,
    "hspace": 0.42,
    "gs_b_wspace": 0.38,
    "gs_heat_wspace": 0.22,
    "gs_heat_hspace": 0.12,
    "bd_shift_left": 0.028,
    "forest_shift_left": 0.024,
    "heat_shift_left": 0.022,
    "heat_shift_right": 0.014,
    "heat_shift_down": 0.032,
    "heat_extra_down": 0.038,
    "heat_scale": 1.21,
    "heat_shift_up": 0.034,
    "heat_top_extra_up": 0.010,
    "heat_nudge_down": 0.022,
    "heat_bottom_extra_down": 0.028,
    "b_top_label_rot": 45,
    "b_top_label_dx": -6.0,
    "b_cont_label_dx": -12.0,
    "gs_col_ratio_right": 1.45,
    "d_axes_width_scale": 1.55,
    "forest_xlabel_dx": -0.018,
    "heat_bottom_shift_down": 0.022,
    "gs_b_ratio_left": 1.15,
    "gs_b_ratio_right": 1.05,
    "ad_forest_height_scale": 1.2,
    "a_label_x": -0.236,
    "a_label_y": 1.02,
    "ab_label_up": -0.001,
    "ac_label_dx": 0.014,
    "cd_label_down": 0.022,
    "a_sn_star_dx": 0.028,
    "a_xlim": (-0.8, 0.8),
    "a_names_x_data": -0.8,
    "a_x_tick_step": 0.4,
    "forest_x_pad_frac": 0.12,
    "forest_x_spine_lim": 0.7,
    "forest_title_x": -0.30,
    "forest_title_shift_left_chars": 4,
    "forest_title_shift_right_chars": 3,
    "forest_title_char_dx_axes": 0.018,
    "forest_names_x_data": -0.7,
    "c_axis_lim": 3.0,
    "c_x_min": -2.6,
    "c_x_max": 2.5,
    "c_x_shift_pt": 10,
    "c_tick_len_pt": 4,
    "c_axis_break": 0.22,
    "c_x_shift": 0.38,
    "b_label_x": -0.14,
    "b_label_y": 0.9922,
    "b_xtick_rot": 22.65,
    "c_leg_gap_chars": 0.15,
    "c_x_tick_start": -2.6,
    "c_x_tick_stop": 2.5,
    "c_x_tick_step": 1.7,
    "c_int_x": 0.97,
    "c_int_y": 0.9658,
    "c_label_x": -0.236,
    "c_label_y": 1.0438,
    "d_path_a_stat_dy": 0.52,
    "d_path_a_lbl_pad": 0.14,
    "d_note_line_dy": 0.68,
    "d_note_high_extra": 0.14,
    "d_path_b_lbl_y": 6.998,
    "d_path_b_stat_y": 6.33,
    "d_int_x": 7.29,
    "d_int_y": 3.92,
    "d_imm_x": 9.55,
    "d_imm_y": 8.45,
    "d_sum_x": 9.55,
    "d_sum_y": 0.35,
    "d_foot_y": 0.11,
    "d_beta_y_offset": 0.34,
    "d_w_box_dy": -1.38,
    "d_int_label_dx": 0.08,
    "d_int_arrow_gap": 0.18,
    "d_left_boxes_dx": 0.45,
    "d_first_box_dx": 0.28,
    "d_content_dx_bias": 0.0,
    "d_box_alpha": 0.52,
    "bd_label_dx": 0.0,
    "d_label_x": 0.02,
    "d_label_y": 1.048
}
WIDTH_MM, HEIGHT_MM = 250, 182
D_BOX_SCALE = 1.5
D_GAP_SCALE = 2.0
D_GAP_SPACING_SCALE = 1.2
D_BOX_W_SCALE = 0.714
D_BOX_W_NARROW = 0.9
D_BOX_H_SCALE = 1.428
D_XLIM = (0.0, 15.8)
D_YLIM = (0.0, 10.0)
DPI = 600
FIG_SAVE_ALPHA_THRESH = 12
FIG_SAVE_BORDER_PX = 3

SN_RED = "#D55E00"
FP_GRAY = "#BDBDBD"
NET_GRAY = "#737373"
BLUE = "#2166AC"
BLUE_LIGHT = "#D1E5F0"
CD_GREEN = "#009B77"
CD_GREEN_LIGHT = "#D0EDE6"
GRAY_LIGHT = "#E8E8E8"
TEXT = "#1A1A1A"
FS = 10
FS_TICK = 9
FS_PANEL = 13
FS_SMALL = 8.5
FS_D_TICK = FS_TICK + 2
FS_D_SMALL = FS_SMALL + 2
FS_D_PANEL = FS_PANEL + 2
FS_D_DIAG_TICK = FS_D_TICK - 1
FS_D_DIAG_SMALL = FS_D_SMALL - 1
SIG_STAR_FS = 10.5
SIG_STAR_PAD_A = 0.03
SIG_STAR_PAD_B = 0.045
SIG_STAR_DY_A = 0.12
SIG_STAR_DY_B = -0.1
FOREST_XLABEL = "Interaction $\\beta$ ($\\Delta$Modularity $\\times$ Reserve)"

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": FS,
        "axes.linewidth": 0.75,
        "axes.labelsize": FS,
        "xtick.labelsize": FS_TICK,
        "ytick.labelsize": FS_TICK,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }
)

NET_ORDER = ["Cont", "Default", "DorsAttn", "SalVentAttn", "SomMot", "Vis"]
SN_NET = "SalVentAttn"
NET_PLOT_LABELS = [
    "Cont",
    "Default",
    "DorsAttn",
    "SN",
    "SomMot",
    "Vis",
]
PC_COLS = [f"net_{n}_participation_coefficient_auc" for n in NET_ORDER]
# Representative subject for panel-b FC heatmaps (None = first entry in npz file).
FC_DISPLAY_SUBJECT: str | None = None
_FC_DATA_CACHE: dict = {}





def fit_lesion_moderation(df: pd.DataFrame) -> dict:
    x_col = "global_modularity_minus_lesion_net_SalVentAttn_global_modularity_auc"
    cols = [x_col, "rt_duration_diff__long-short", "rt_transition_diff__flexibility_stability"]
    sub = df[cols].dropna().copy()
    for c in cols:
        sub[c] = zscore_series(sub[c])
    x, w, y = (sub[c].values for c in cols)
    model = OLS(y, add_constant(np.column_stack([x, w, x * w]))).fit()
    return {"model": model, "sub_z": sub}


def fit_all_network_lesion_interactions(df: pd.DataFrame) -> pd.DataFrame:
    """ΔModularity (per network lesion) × adaptive reserve → behavior; interaction β₃ per network."""
    y_col = "rt_transition_diff__flexibility_stability"
    w_col = "rt_duration_diff__long-short"
    rows = []
    for net in NET_ORDER:
        x_col = f"global_modularity_minus_lesion_net_{net}_global_modularity_auc"
        cols = [x_col, w_col, y_col]
        sub = df[cols].dropna().copy()
        for c in cols:
            sub[c] = zscore_series(sub[c])
        x, w, y = (sub[c].values for c in cols)
        model = OLS(y, add_constant(np.column_stack([x, w, x * w]))).fit()
        ci = np.asarray(model.conf_int())[3]
        rows.append(
            {
                "network": net,
                "b3": float(model.params[3]),
                "ci_lo": float(ci[0]),
                "ci_hi": float(ci[1]),
                "is_sn": net == "SalVentAttn",
                "is_vis": net == "Vis",
            }
        )
    return pd.DataFrame(rows)


def _reorder_fc_by_network(
    fc: np.ndarray, region_names: np.ndarray
) -> tuple[np.ndarray, dict[str, tuple[int, int]]]:
    """Sort parcels by canonical network so each network forms one contiguous block."""
    order: list[int] = []
    bounds: dict[str, tuple[int, int]] = {}
    start = 0
    for net in NET_ORDER:
        idx = np.where(region_names == net)[0]
        order.extend(idx.tolist())
        end = start + len(idx)
        bounds[net] = (start, end)
        start = end
    order_arr = np.asarray(order, dtype=int)
    return fc[np.ix_(order_arr, order_arr)], bounds


def _load_schematic_fc_matrices(
    cache_key: str,
) -> tuple[np.ndarray, np.ndarray, tuple[int, int], str, dict[str, tuple[int, int]]]:
    """Block-structured schematic FC for panel-b when no FC file is available."""
    n_total = 1000
    n_base, remainder = divmod(n_total, len(NET_ORDER))
    rng = np.random.RandomState(42)
    fc = np.zeros((n_total, n_total), dtype=np.float64)
    bounds: dict[str, tuple[int, int]] = {}
    start = 0
    for i, net in enumerate(NET_ORDER):
        n = n_base + (1 if i < remainder else 0)
        end = start + n
        bounds[net] = (start, end)
        block = rng.uniform(0.35, 0.75, (n, n))
        block = (block + block.T) / 2.0
        np.fill_diagonal(block, 1.0)
        fc[start:end, start:end] = block
        start = end
    between = rng.uniform(0.05, 0.18, (n_total, n_total))
    between = (between + between.T) / 2.0
    for i0, i1 in bounds.values():
        between[i0:i1, i0:i1] = 0.0
    fc += between
    np.fill_diagonal(fc, 1.0)
    i0, i1 = bounds[SN_NET]
    lesioned = fc.copy()
    lesioned[i0:i1, :] = 0.0
    lesioned[:, i0:i1] = 0.0
    warnings.warn(
        "FC matrix file not found; panel-b heatmaps use schematic block-structured matrices. "
        "Add fc_display_matrices.npz or per_subject_fc.npz under data/ (see data/README.md) "
        "for participant-derived connectivity.",
        UserWarning,
        stacklevel=3,
    )
    _FC_DATA_CACHE[cache_key] = {
        "intact": fc.astype(np.float32),
        "lesioned": lesioned.astype(np.float32),
        "sn_block": (i0, i1),
        "bounds": bounds,
        "subject": "schematic",
    }
    c = _FC_DATA_CACHE[cache_key]
    return c["intact"], c["lesioned"], c["sn_block"], c["subject"], c["bounds"]


def _load_display_fc_matrices(
    subject_id: str | None = None,
) -> tuple[np.ndarray, np.ndarray, tuple[int, int], str]:
    """Single-subject parcel FC, network-block ordered; SN lesion = one contiguous block."""
    cache_key = subject_id or "__default__"
    if cache_key in _FC_DATA_CACHE:
        c = _FC_DATA_CACHE[cache_key]
        return c["intact"], c["lesioned"], c["sn_block"], c["subject"], c["bounds"]

    try:
        fc_path = _resolve_fc_npz()
    except FileNotFoundError:
        return _load_schematic_fc_matrices(cache_key)

    z = np.load(fc_path, allow_pickle=True)
    if "intact_fc" in z.files and "lesioned_fc" in z.files:
        region_names = np.asarray(z["region_names"])
        fc_ord = np.asarray(z["intact_fc"], dtype=np.float64)
        lesioned = np.asarray(z["lesioned_fc"], dtype=np.float64)
        bounds_raw = z["network_bounds"].item() if "network_bounds" in z.files else None
        z.close()
        if bounds_raw is not None:
            bounds = {k: tuple(v) for k, v in bounds_raw.items()}
        else:
            _, bounds = _reorder_fc_by_network(fc_ord, region_names)
        i0, i1 = bounds[SN_NET]
        _FC_DATA_CACHE[cache_key] = {
            "intact": fc_ord.astype(np.float32),
            "lesioned": lesioned.astype(np.float32),
            "sn_block": (i0, i1),
            "bounds": bounds,
            "subject": "group_mean",
        }
        return (
            _FC_DATA_CACHE[cache_key]["intact"],
            _FC_DATA_CACHE[cache_key]["lesioned"],
            _FC_DATA_CACHE[cache_key]["sn_block"],
            _FC_DATA_CACHE[cache_key]["subject"],
            _FC_DATA_CACHE[cache_key]["bounds"],
        )
    per_subject: dict = z["per_subject_FC"].item()
    region_names = np.asarray(z["region_names"])
    z.close()

    if subject_id is None:
        subject_id = sorted(per_subject.keys())[0]
    if subject_id not in per_subject:
        raise KeyError(f"Subject {subject_id!r} not in {fc_path.name}")

    fc = np.asarray(per_subject[subject_id], dtype=np.float64)
    np.fill_diagonal(fc, 1.0)
    fc_ord, bounds = _reorder_fc_by_network(fc, region_names)
    i0, i1 = bounds[SN_NET]
    lesioned = fc_ord.copy()
    lesioned[i0:i1, :] = 0.0
    lesioned[:, i0:i1] = 0.0

    _FC_DATA_CACHE[cache_key] = {
        "intact": fc_ord.astype(np.float32),
        "lesioned": lesioned.astype(np.float32),
        "sn_block": (i0, i1),
        "bounds": bounds,
        "subject": str(subject_id),
    }
    return (
        _FC_DATA_CACHE[cache_key]["intact"],
        _FC_DATA_CACHE[cache_key]["lesioned"],
        _FC_DATA_CACHE[cache_key]["sn_block"],
        _FC_DATA_CACHE[cache_key]["subject"],
        _FC_DATA_CACHE[cache_key]["bounds"],
    )


def _remove_letter_text(ax: plt.Axes, letter: str) -> None:
    for text in list(ax.texts):
        if text.get_text().strip() == letter:
            text.remove()


def _remove_figure_letter_text(fig: plt.Figure, letter: str) -> None:
    for text in list(fig.texts):
        if text.get_text().strip() == letter:
            text.remove()


def _compute_panel_d_layout(content_dx: float | None = None) -> dict:
    """Shared D-panel box geometry (data coords before centering shift)."""
    s, g = D_BOX_SCALE, D_GAP_SCALE
    w_scale = D_BOX_W_SCALE * D_BOX_W_NARROW
    h_scale = D_BOX_H_SCALE
    box_w_scale = 1.1 * 0.9
    bw_x = 2.4 * s * w_scale * box_w_scale
    bh = 1.1 * s * h_scale
    bw_y = 2.5 * s * w_scale * box_w_scale
    bh_y = 1.1 * s * h_scale
    bw_w = 2.6 * s * w_scale * box_w_scale
    bh_w = 1.0 * s * h_scale
    gap_h = 0.8 * g * D_GAP_SPACING_SCALE
    gap_v = 1.0 * g * D_GAP_SPACING_SCALE
    y_top = 6.0
    x0_x = 0.6 - LAYOUT["d_left_boxes_dx"] - LAYOUT["d_first_box_dx"]
    x0_m = x0_x + bw_x + gap_h
    x0_y = x0_m + bw_x + gap_h
    y0_w = y_top - gap_v - bh_w + LAYOUT["d_w_box_dy"]
    x0_w = (x0_m + bw_x / 2 + x0_y + bw_y / 2 - bw_w) / 2
    content_left = min(x0_x, x0_w)
    content_right = x0_y + bw_y
    content_bottom = y0_w - 0.35
    imm_y = y_top + bh + 0.65
    content_top = imm_y + 0.55
    content_cx = (content_left + content_right) / 2
    content_cy = (content_bottom + content_top) / 2
    dx = LAYOUT["d_content_dx_bias"] if content_dx is None else content_dx
    ox = (D_XLIM[0] + D_XLIM[1]) / 2 - content_cx + dx
    oy = (D_YLIM[0] + D_YLIM[1]) / 2 - content_cy
    return {
        "x0_x": x0_x,
        "x0_m": x0_m,
        "x0_y": x0_y,
        "x0_w": x0_w,
        "y_top": y_top,
        "y0_w": y0_w,
        "bw_x": bw_x,
        "bh": bh,
        "bw_y": bw_y,
        "bh_y": bh_y,
        "bw_w": bw_w,
        "bh_w": bh_w,
        "gap_h": gap_h,
        "content_left": content_left,
        "ox": ox,
        "oy": oy,
        "path_b_mid_x": (x0_m + bw_x + x0_y) / 2,
        "content_cx": content_cx,
        "content_cy": content_cy,
    }


def _d_content_dx_aligned_to_b_heat_yticklabels(
    fig: plt.Figure,
    ax_d: plt.Axes,
    ax_htop: plt.Axes,
    ax_hbot: plt.Axes,
) -> float:
    """Offset D content so leftmost box left edge aligns with B heatmap y tick labels."""
    fig.canvas.draw()
    target_x = min(
        _fig_x_from_axes_yaxis_ticklabels_left(fig, ax_htop),
        _fig_x_from_axes_yaxis_ticklabels_left(fig, ax_hbot),
    )
    lay_ref = _compute_panel_d_layout(0.0)
    left_data_ref = lay_ref["content_left"] + lay_ref["ox"]
    pos = ax_d.get_position()
    xlim_w = D_XLIM[1] - D_XLIM[0]
    dx = (target_x - pos.x0) / pos.width * xlim_w + D_XLIM[0] - left_data_ref
    return dx + LAYOUT["d_content_dx_bias"]


def _fig_x_panel_d_leftmost_box(
    fig: plt.Figure,
    ax_d: plt.Axes,
    content_dx: float | None = None,
) -> float:
    """Figure x of left border of the leftmost D-panel rectangle."""
    lay = _compute_panel_d_layout(content_dx)
    fig.canvas.draw()
    x_data = lay["content_left"] + lay["ox"]
    y_data = (D_YLIM[0] + D_YLIM[1]) / 2
    return float(fig.transFigure.inverted().transform(ax_d.transData.transform((x_data, y_data)))[0])


def _place_panel_labels(
    fig: plt.Figure,
    ax_a: plt.Axes,
    ax_htop: plt.Axes,
    ax_c: plt.Axes,
    ax_d: plt.Axes,
    *,
    d_content_dx: float | None = None,
) -> None:
    """panel a/c left; b/d left-aligned to mediation box; ab top-aligned."""
    fig.canvas.draw()
    y_ab = float(
        fig.transFigure.inverted().transform(
            ax_a.transAxes.transform((LAYOUT["a_label_x"], LAYOUT["a_label_y"]))
        )[1]
    ) + LAYOUT["ab_label_up"]
    x_ac = float(
        fig.transFigure.inverted().transform(
            ax_c.transAxes.transform((LAYOUT["c_label_x"], 0.0))
        )[0]
    )
    x_bd = _fig_x_panel_d_leftmost_box(fig, ax_d, d_content_dx) + LAYOUT["bd_label_dx"]
    y_cd = float(
        fig.transFigure.inverted().transform(
            ax_c.transAxes.transform((0.0, LAYOUT["c_label_y"]))
        )[1]
    )

    for ax, letter in ((ax_a, "a"), (ax_c, "c"), (ax_d, "d")):
        _remove_letter_text(ax, letter)
    for letter in ("a", "b", "c", "d"):
        _remove_figure_letter_text(fig, letter)

    label_kw = dict(fontsize=FS_PANEL, fontweight="bold", transform=fig.transFigure)
    fig.text(x_ac, y_ab, "a", ha="left", va="top", **label_kw)
    fig.text(x_bd, y_ab, "b", ha="right", va="top", **label_kw)
    fig.text(x_ac, y_cd, "c", ha="left", va="top", **label_kw)
    fig.text(x_bd, y_cd, "d", ha="right", va="top", **label_kw)


def _fig_x_from_axes_yaxis_ticklabels_left(fig: plt.Figure, ax: plt.Axes) -> float:
    fig.canvas.draw()
    bb = ax.yaxis.get_tightbbox(fig.canvas.get_renderer())
    return float(fig.transFigure.inverted().transform((bb.x0, bb.y0))[0])


def _nudge_heatmaps_to_forest_xlabel(
    fig: plt.Figure,
    heat_axes: list[plt.Axes],
    ax_hbot: plt.Axes,
    ax_a: plt.Axes,
) -> None:
    """Shift heatmaps down so bottom title baseline matches forest x-axis title."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    bb_a = ax_a.xaxis.label.get_window_extent(renderer)
    y_forest_bottom = float(fig.transFigure.inverted().transform((bb_a.x0, bb_a.y0))[1])
    title_bottom = None
    for text in ax_hbot.texts:
        x_ax, y_ax = text.get_position()
        if text.get_transform() != ax_hbot.transAxes or y_ax >= 0:
            continue
        bb = text.get_window_extent(renderer)
        title_bottom = float(fig.transFigure.inverted().transform((bb.x0, bb.y0))[1])
        break
    if title_bottom is None:
        return
    dy = y_forest_bottom - title_bottom
    if abs(dy) > 1e-5:
        _nudge_axes_positions(heat_axes, dy=dy)


def _nudge_axes_positions(axes: list[plt.Axes], dx: float = 0.0, dy: float = 0.0) -> None:
    for ax in axes:
        pos = ax.get_position()
        ax.set_position([pos.x0 + dx, pos.y0 + dy, pos.width, pos.height])


def _scale_axes(axes: list[plt.Axes], scale: float, dy: float = 0.0) -> None:
    for ax in axes:
        pos = ax.get_position()
        cx = pos.x0 + pos.width / 2
        cy = pos.y0 + pos.height / 2
        w = pos.width * scale
        h = pos.height * scale
        ax.set_position([cx - w / 2, cy - h / 2 + dy, w, h])


def _stretch_axes_height(axes: list[plt.Axes], scale: float) -> None:
    """Increase axes height only (width unchanged); vertically centered in place."""
    for ax in axes:
        pos = ax.get_position()
        cy = pos.y0 + pos.height / 2
        h = pos.height * scale
        ax.set_position([pos.x0, cy - h / 2, pos.width, h])


def _forest_point_style(row) -> tuple[str, float]:
    if row.is_sn:
        return SN_RED, 5.5
    if row.is_vis:
        return FP_GRAY, 5.0
    return NET_GRAY, 5.5


def _sig_stars(p: float) -> str:
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def _capitalize_words(text: str) -> str:
    """Title-case each word; preserve newlines and leading punctuation."""
    lines: list[str] = []
    for line in text.split("\n"):
        words: list[str] = []
        for word in line.split(" "):
            if not word:
                words.append(word)
                continue
            idx = next((i for i, ch in enumerate(word) if ch.isalpha()), len(word))
            if idx < len(word):
                word = word[:idx] + word[idx].upper() + word[idx + 1 :]
            words.append(word)
        lines.append(" ".join(words))
    return "\n".join(lines)


def _hide_y_spine_and_tick_marks(ax: plt.Axes) -> None:
    """Keep y tick labels; remove left spine and y tick marks."""
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0, width=0, pad=5, colors=TEXT)


def _panel_c_x_shift(ax: plt.Axes, shift_pt: float | None = None) -> float:
    """Rightward data shift for X (gap between Y- and X-axis lines), per moderate_figure."""
    x_min = LAYOUT["c_x_min"]
    y_min = -LAYOUT["c_axis_lim"]
    shift_pt = LAYOUT["c_x_shift_pt"] if shift_pt is None else shift_pt
    ax.figure.canvas.draw()
    p0 = ax.transData.transform((x_min, y_min))
    scale = ax.figure.dpi / 72.0
    gx, _ = ax.transData.inverted().transform((p0[0] + shift_pt * scale, p0[1]))
    return float(gx - x_min)


def _panel_c_tick_len_data(ax: plt.Axes) -> float:
    y_min = -LAYOUT["c_axis_lim"]
    p0 = ax.transData.transform((LAYOUT["c_x_min"], y_min))
    p1 = (p0[0], p0[1] - LAYOUT["c_tick_len_pt"] * ax.figure.dpi / 72.0)
    _, y1 = ax.transData.inverted().transform(p1)
    return float(y_min - y1)


def _panel_c_draw_disconnected_axes(ax: plt.Axes, x_shift: float) -> None:
    """Hide spines; draw Y at xmin and X from xmin+x_shift (moderate_figure style)."""
    xmin = LAYOUT["c_x_min"]
    xmax = LAYOUT["c_x_max"]
    ymin, ymax = -LAYOUT["c_axis_lim"], LAYOUT["c_axis_lim"]
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(
        axis="both",
        top=False,
        right=False,
        bottom=True,
        left=True,
        direction="out",
        length=0,
        width=0.8,
        colors=TEXT,
        pad=8,
    )
    kw = dict(color=TEXT, lw=0.9, clip_on=False, zorder=6, solid_capstyle="butt")
    ax.plot([xmin, xmin], [ymin, ymax], **kw)
    ax.plot([xmin + x_shift, xmax + x_shift], [ymin, ymin], **kw)


def _panel_c_outward_tick_marks(ax: plt.Axes) -> None:
    xmin = LAYOUT["c_x_min"]
    ymin = -LAYOUT["c_axis_lim"]
    tl = _panel_c_tick_len_data(ax)
    tk = dict(color=TEXT, lw=0.8, clip_on=False, zorder=8, solid_capstyle="butt")
    for x in ax.get_xticks():
        ax.plot([x, x], [ymin, ymin - tl], **tk)
    for y in ax.get_yticks():
        ax.plot([xmin - tl, xmin], [y, y], **tk)


def _panel_c_legend_bbox_anchor(
    ax: plt.Axes, x_shift: float
) -> tuple[tuple[float, float], mpl.transforms.Transform]:
    """Lower-center legend anchor one character above the X axis."""
    x_min = LAYOUT["c_x_min"]
    x_max = LAYOUT["c_x_max"]
    y_min = -LAYOUT["c_axis_lim"]
    x_center = (x_min + x_max + x_shift) / 2.0
    ax.figure.canvas.draw()
    px, py = ax.transData.transform((x_center, y_min))
    char_px = FS_SMALL * LAYOUT["c_leg_gap_chars"] * ax.figure.dpi / 72.0
    _, y_anchor = ax.transData.inverted().transform((px, py + char_px))
    return (x_center, y_anchor), ax.transData


def _forest_symmetric_xlim(plot_df: pd.DataFrame) -> tuple[float, float, float]:
    ext = float(
        max(plot_df["ci_hi"].max(), abs(plot_df["ci_lo"].min()), abs(plot_df["b3"]).max())
    )
    half = max(0.45, float(np.ceil(ext * 10) / 10))
    if half > 0.55:
        half = 0.7
    pad = half * LAYOUT["forest_x_pad_frac"]
    return -half - pad, half + pad * 1.35, half


def _annotate_sig_star(ax, x_right: float, y: float, stars: str, color: str, dy: float = 0.0) -> None:
    ax.text(
        x_right,
        y + dy,
        stars,
        va="center",
        ha="left",
        fontsize=SIG_STAR_FS,
        color=color,
        fontweight="bold",
        clip_on=False,
    )


def _conditional_line_band(
    xg: np.ndarray, w_fix: float, model, alpha: float = 0.95
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    params = np.asarray(model.params, dtype=float)
    cov = np.asarray(model.cov_params(), dtype=float)
    tc = float(scipy_stats.t.ppf(0.5 + alpha / 2, model.df_resid))
    yhat = np.empty_like(xg)
    lo = np.empty_like(xg)
    hi = np.empty_like(xg)
    for i, xv in enumerate(xg):
        grad = np.array([1.0, xv, w_fix, xv * w_fix])
        pred = float(grad @ params)
        se = float(np.sqrt(grad @ cov @ grad))
        yhat[i] = pred
        lo[i] = pred - tc * se
        hi[i] = pred + tc * se
    return yhat, lo, hi


def _annotate_panel_a_network_labels(ax: plt.Axes, y_pos: np.ndarray) -> None:
    """Network names right-aligned to left x-tick at -0.8."""
    names_x = LAYOUT["a_names_x_data"]
    for yi, name in zip(y_pos, NET_PLOT_LABELS, strict=False):
        ax.text(
            names_x,
            yi,
            _capitalize_words(name),
            transform=ax.transData,
            ha="right",
            va="center",
            fontsize=FS_TICK,
            color=TEXT,
            clip_on=False,
            zorder=5,
        )


def panel_a(ax, model) -> None:
    y_pos = np.arange(len(PC_COLS))
    betas, lo, hi, ps, colors = [], [], [], [], []
    for i, col in enumerate(PC_COLS):
        betas.append(float(model.params.iloc[i + 1]))
        ci = model.conf_int().iloc[i + 1]
        lo.append(float(ci[0]))
        hi.append(float(ci[1]))
        p = float(model.pvalues.iloc[i + 1])
        ps.append(p)
        if col.endswith("SalVentAttn_participation_coefficient_auc"):
            colors.append(SN_RED)
        elif col.endswith("Cont_participation_coefficient_auc"):
            colors.append(FP_GRAY)
        else:
            colors.append(NET_GRAY)
    ax.barh(y_pos, betas, color=colors, height=0.62, zorder=3, edgecolor="none")
    ax.errorbar(
        betas,
        y_pos,
        xerr=[np.array(betas) - np.array(lo), np.array(hi) - np.array(betas)],
        fmt="none",
        ecolor=TEXT,
        capsize=2.5,
        lw=0.9,
        zorder=4,
    )
    ax.axvline(0, color=TEXT, lw=0.6, zorder=2)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([])
    ax.set_xlabel(_capitalize_words("Standardized Regression Coefficient (Beta weights)"))
    ax.invert_yaxis()
    x0, x1 = LAYOUT["a_xlim"]
    step = LAYOUT["a_x_tick_step"]
    ax.set_xlim(x0, x1)
    ax.set_xticks(np.arange(x0, x1 + step * 0.5, step))
    _hide_y_spine_and_tick_marks(ax)
    _annotate_panel_a_network_labels(ax, y_pos)
    for i, p in enumerate(ps):
        stars = _sig_stars(p)
        if not stars:
            continue
        x_right = max(hi[i], lo[i]) + SIG_STAR_PAD_A
        col = SN_RED if i == PC_COLS.index(f"net_{SN_NET}_participation_coefficient_auc") else "#666"
        _annotate_sig_star(ax, x_right, i, stars, col, dy=SIG_STAR_DY_A)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def _network_tick_centers(bounds: dict[str, tuple[int, int]]) -> list[float]:
    return [(bounds[net][0] + bounds[net][1]) / 2.0 - 0.5 for net in NET_ORDER]


def _draw_fc_network_grid(ax, bounds: dict[str, tuple[int, int]]) -> None:
    for net in NET_ORDER[1:]:
        edge = bounds[net][0] - 0.5
        ax.axhline(edge, color="white", ls=(0, (2.5, 2.5)), lw=0.7, zorder=5)
        ax.axvline(edge, color="white", ls=(0, (2.5, 2.5)), lw=0.7, zorder=5)


def _annotate_fc_network_axes(
    ax,
    bounds: dict[str, tuple[int, int]],
    *,
    top_x_labels: bool = False,
    left_y_labels: bool = True,
) -> None:
    centers = _network_tick_centers(bounds)
    if left_y_labels:
        ax.set_yticks(centers)
        ax.set_yticklabels(NET_PLOT_LABELS, fontsize=FS_TICK)
    else:
        ax.set_yticks([])
    if top_x_labels:
        ax.xaxis.tick_top()
        x_centers = [c + LAYOUT["b_top_label_dx"] for c in centers]
        x_centers[0] += LAYOUT["b_cont_label_dx"]
        ax.set_xticks(x_centers)
        ax.set_xticklabels(
            NET_PLOT_LABELS,
            rotation=LAYOUT["b_top_label_rot"],
            ha="left",
            va="bottom",
            fontsize=FS_TICK,
        )
    else:
        ax.set_xticks([])
    ax.tick_params(axis="both", length=0, pad=1.2, colors=TEXT)


def _fc_colormap_limits(fc: np.ndarray) -> float:
    """Symmetric diverging limits from off-diagonal |r|."""
    off = fc.copy()
    np.fill_diagonal(off, np.nan)
    vmax = float(np.nanpercentile(np.abs(off), 99))
    return float(np.clip(vmax, 0.38, 0.62))


def _fc_heatmap(
    ax,
    fc: np.ndarray,
    bounds: dict[str, tuple[int, int]],
    sn_block: tuple[int, int],
    lesioned: bool,
    title: str,
    vmax: float,
    *,
    top_x_labels: bool = False,
    left_y_labels: bool = True,
    title_at_bottom: bool = False,
) -> None:
    """Parcel FC (network-block ordered); lesioned panel shows one SN block only."""
    n = fc.shape[0]
    plot_fc = fc.astype(np.float64, copy=True)
    np.fill_diagonal(plot_fc, np.nan)
    if lesioned:
        i0, i1 = sn_block
        plot_fc[i0:i1, i0:i1] = np.nan
    cmap = plt.get_cmap("RdBu_r").copy()
    cmap.set_bad("#F7F7F7")
    ax.imshow(
        plot_fc,
        cmap=cmap,
        vmin=-vmax,
        vmax=vmax,
        aspect="equal",
        interpolation="nearest",
    )
    _draw_fc_network_grid(ax, bounds)
    if lesioned:
        i0, i1 = sn_block
        if i0 > 0:
            ax.plot(
                [-0.5, i0 - 0.5],
                [-0.5, i0 - 0.5],
                color="#B2182B",
                lw=0.45,
                zorder=6,
                solid_capstyle="round",
            )
        if i1 < n:
            ax.plot(
                [i1 - 0.5, n - 0.5],
                [i1 - 0.5, n - 0.5],
                color="#B2182B",
                lw=0.45,
                zorder=6,
                solid_capstyle="round",
            )
    else:
        ax.plot(
            [-0.5, n - 0.5],
            [-0.5, n - 0.5],
            color="#B2182B",
            lw=0.45,
            zorder=6,
            solid_capstyle="round",
        )
    _annotate_fc_network_axes(
        ax, bounds, top_x_labels=top_x_labels, left_y_labels=left_y_labels
    )
    for spine in ax.spines.values():
        spine.set_visible(False)
    if lesioned:
        i0, i1 = sn_block
        ax.add_patch(
            mpatches.Rectangle(
                (i0 - 0.5, i0 - 0.5),
                i1 - i0,
                i1 - i0,
                fill=False,
                edgecolor=SN_RED,
                lw=1.0,
                linestyle="--",
            )
        )
    if title_at_bottom:
        ax.text(
            0.5,
            -0.055,
            title,
            transform=ax.transAxes,
            ha="center",
            va="top",
            fontsize=FS_TICK,
            clip_on=False,
        )
    else:
        ax.set_title(title, fontsize=FS_TICK, pad=3)


def _annotate_forest_y_labels(ax: plt.Axes, plot_df: pd.DataFrame, y: np.ndarray) -> None:
    """Y title left; network names right-aligned to x = -0.7 (left x-tick / spine start)."""
    char_dx = LAYOUT["forest_title_char_dx_axes"]
    title_x = (
        LAYOUT["forest_title_x"]
        - LAYOUT["forest_title_shift_left_chars"] * char_dx
        + LAYOUT["forest_title_shift_right_chars"] * char_dx
    )
    ax.text(
        title_x,
        0.5,
        "Virtual Lesion Network",
        transform=ax.transAxes,
        rotation=90,
        ha="center",
        va="center",
        fontsize=FS,
        color=TEXT,
        clip_on=False,
        zorder=5,
    )
    names_x = LAYOUT["forest_names_x_data"]
    for yi, row in zip(y, plot_df.itertuples(), strict=False):
        ax.text(
            names_x,
            yi,
            "SN" if row.network == SN_NET else row.network,
            transform=ax.transData,
            ha="right",
            va="center",
            fontsize=FS_TICK,
            color=TEXT,
            clip_on=False,
            zorder=5,
        )


def _plot_interaction_forest(ax, inter_df: pd.DataFrame) -> None:
    """Forest: diamonds, symmetric x-ticks; custom y title / network name positions."""
    plot_df = inter_df.iloc[::-1].reset_index(drop=True)
    y = np.arange(len(plot_df))
    x_lo, x_hi, x_half = _forest_symmetric_xlim(plot_df)
    for yi, row in zip(y, plot_df.itertuples(), strict=False):
        col, ms = _forest_point_style(row)
        ax.errorbar(
            row.b3,
            yi,
            xerr=[[row.b3 - row.ci_lo], [row.ci_hi - row.b3]],
            fmt="D",
            ms=ms,
            mew=0.6,
            color=col,
            markerfacecolor=col,
            markeredgecolor=col,
            ecolor=col,
            capsize=0,
            lw=0.95,
            zorder=3,
        )
    ax.axvline(0, color=TEXT, ls=(0, (4.0, 3.0)), lw=0.75, zorder=1)
    ax.set_yticks(y)
    ax.set_yticklabels([])
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.set_xlim(x_lo, x_hi)
    spine_lim = LAYOUT["forest_x_spine_lim"]
    ax.set_xticks([-spine_lim, 0.0, spine_lim])
    ax.tick_params(axis="x", pad=4, length=3.5, width=0.75, colors=TEXT)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_bounds(-spine_lim, spine_lim)
    _hide_y_spine_and_tick_marks(ax)
    _annotate_forest_y_labels(ax, plot_df, y)


def _add_forest_xlabel(fig: plt.Figure, ax_forest: plt.Axes, ax_a: plt.Axes) -> None:
    """Figure-level xlabel; bottom edge aligned with panel A x-axis title."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    bb_a = ax_a.xaxis.label.get_window_extent(renderer=renderer)
    y_label = fig.transFigure.inverted().transform((bb_a.x0, bb_a.y0))[1]
    pos = ax_forest.get_position()
    x_center = (pos.x0 + pos.x1) / 2 + LAYOUT["forest_xlabel_dx"]
    fig.text(
        x_center,
        y_label,
        FOREST_XLABEL,
        transform=fig.transFigure,
        ha="center",
        va="bottom",
        fontsize=FS,
        color=TEXT,
        clip_on=False,
    )


def _match_forest_axes_to_a(ax_a: plt.Axes, ax_forest: plt.Axes) -> None:
    """Match forest plot data-area height and vertical extent to panel A."""
    pos_a = ax_a.get_position()
    pos_f = ax_forest.get_position()
    ax_forest.set_position([pos_f.x0, pos_a.y0, pos_f.width, pos_a.height])


def panel_b(ax_heat_top, ax_heat_bottom, ax_forest, df: pd.DataFrame) -> None:
    intact_fc, lesioned_fc, sn_block, _subj_id, bounds = _load_display_fc_matrices(
        FC_DISPLAY_SUBJECT
    )
    vmax = _fc_colormap_limits(intact_fc)
    _fc_heatmap(
        ax_heat_top,
        intact_fc,
        bounds,
        sn_block,
        False,
        "Intact Connectome",
        vmax,
        top_x_labels=False,
        left_y_labels=True,
        title_at_bottom=True,
    )
    _fc_heatmap(
        ax_heat_bottom,
        lesioned_fc,
        bounds,
        sn_block,
        True,
        "Virtual Lesioned SN",
        vmax,
        top_x_labels=False,
        left_y_labels=True,
        title_at_bottom=True,
    )
    inter_df = fit_all_network_lesion_interactions(df)
    _plot_interaction_forest(ax_forest, inter_df)


class _BandLineLegendHandle:
    def __init__(self, line_color: str, band_color: str):
        self.line_color = line_color
        self.band_color = band_color


class _BandLineLegendHandler(HandlerBase):
    """Legend icon: shaded square with a horizontal line through the center."""

    def create_artists(
        self,
        legend,
        orig_handle,
        xdescent,
        ydescent,
        width,
        height,
        fontsize,
        trans,
    ):
        band = Rectangle(
            (xdescent, ydescent),
            width,
            height,
            facecolor=orig_handle.band_color,
            edgecolor="none",
            alpha=0.55,
            transform=trans,
        )
        y_mid = ydescent + height / 2
        line = Line2D(
            [xdescent, xdescent + width],
            [y_mid, y_mid],
            color=orig_handle.line_color,
            lw=2.0,
            solid_capstyle="butt",
            transform=trans,
        )
        return [band, line]


def panel_c(ax, lesion_fit: dict) -> None:
    sub = lesion_fit["sub_z"]
    x_col = "global_modularity_minus_lesion_net_SalVentAttn_global_modularity_auc"
    w_col = "rt_duration_diff__long-short"
    y_col = "rt_transition_diff__flexibility_stability"
    x, w, y = sub[x_col].values, sub[w_col].values, sub[y_col].values
    hi = w <= -0.5
    gray = ~hi
    model = lesion_fit["model"]
    x_min = LAYOUT["c_x_min"]
    x_max = LAYOUT["c_x_max"]
    y_lim = LAYOUT["c_axis_lim"]
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(-y_lim, y_lim)
    ax.figure.canvas.draw()
    x_shift = _panel_c_x_shift(ax)
    xg = np.linspace(x_min, x_max, 160) + x_shift
    band_specs = [
        (-1.0, CD_GREEN, CD_GREEN_LIGHT, "High Reserve"),
        (1.0, NET_GRAY, GRAY_LIGHT, "Low Reserve"),
    ]
    for w_fix, line_c, band_c, _label in band_specs:
        yhat, ylo, yhi = _conditional_line_band(np.linspace(x_min, x_max, 160), w_fix, model)
        ax.fill_between(xg, ylo, yhi, color=band_c, alpha=0.55, zorder=1, linewidth=0)
        ax.plot(xg, yhat, color=line_c, lw=2.0, zorder=4)
    ax.scatter(
        x[gray] + x_shift,
        y[gray],
        s=22,
        c=NET_GRAY,
        alpha=0.6,
        edgecolors="none",
        zorder=2,
    )
    ax.scatter(
        x[hi] + x_shift,
        y[hi],
        s=24,
        c=CD_GREEN,
        alpha=0.8,
        edgecolors="white",
        linewidths=0.35,
        zorder=3,
    )
    ax.set_xlabel(
        "SN-Specific Topological Contribution ($\\Delta$Modularity$_{\\mathrm{SN}}$, $Z$-score)"
    )
    ax.set_ylabel("Behavioral Coordination ($RT_{\\mathrm{Diff}}$, $Z$-score)")
    x_ticks = np.arange(
        LAYOUT["c_x_tick_start"],
        LAYOUT["c_x_tick_stop"] + 1e-9,
        LAYOUT["c_x_tick_step"],
    )
    y_ticks = [-3.0, -1.5, 0.0, 1.5, 3.0]
    ax.set_xticks([t + x_shift for t in x_ticks])
    ax.set_xticklabels(
        [
            f"{t:.0f}" if abs(t - round(t)) < 1e-6 else f"{t:.1f}"
            for t in x_ticks
        ]
    )
    ax.set_yticks(y_ticks)
    ax.set_yticklabels([f"{t:.1f}" for t in y_ticks])
    ax.set_xlim(x_min, x_max + x_shift)
    leg_anchor, leg_transform = _panel_c_legend_bbox_anchor(ax, x_shift)
    leg_handles = [
        _BandLineLegendHandle(CD_GREEN, CD_GREEN_LIGHT),
        _BandLineLegendHandle(NET_GRAY, GRAY_LIGHT),
    ]
    ax.legend(
        handles=leg_handles,
        labels=[spec[3] for spec in band_specs],
        handler_map={_BandLineLegendHandle: _BandLineLegendHandler()},
        frameon=False,
        fontsize=FS_SMALL,
        loc="lower center",
        bbox_to_anchor=leg_anchor,
        bbox_transform=leg_transform,
        handlelength=1.15,
        handleheight=1.15,
        handletextpad=0.55,
        borderaxespad=0.2,
    )
    _panel_c_draw_disconnected_axes(ax, x_shift)
    _panel_c_outward_tick_marks(ax)


def _align_d_footnotes_bottom_to_c_xlabel(
    fig: plt.Figure,
    ax_c: plt.Axes,
    ax_d: plt.Axes,
    *,
    line_dy: float,
) -> None:
    """Align D-panel last footnote bottom edge with C-panel x-axis title bottom."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    bb_c = ax_c.xaxis.label.get_window_extent(renderer)
    y_fig = fig.transFigure.inverted().transform((bb_c.x0, bb_c.y0))[1]
    pos_d = ax_d.get_position()
    y_axes = (y_fig - pos_d.y0) / pos_d.height
    y_low_bottom = ax_d.transData.inverted().transform(ax_d.transAxes.transform((0.5, y_axes)))[1]

    low_text = high_text = None
    for text in ax_d.texts:
        body = text.get_text()
        if body.startswith("Low Reserve"):
            low_text = text
        elif body.startswith("High Reserve"):
            high_text = text

    if low_text is None:
        return

    low_text.set_verticalalignment("bottom")
    low_text.set_y(y_low_bottom)
    if high_text is not None:
        high_text.set_verticalalignment("bottom")
        high_text.set_y(y_low_bottom + line_dy + LAYOUT["d_note_high_extra"])


def panel_d(ax, med: dict, *, content_dx: float | None = None) -> None:
    """Mediation diagram + IMM line centered as one block in panel D."""
    ax.clear()
    ax.set_xlim(*D_XLIM)
    ax.set_ylim(*D_YLIM)
    ax.axis("off")
    lay = _compute_panel_d_layout(content_dx)
    x0_x = lay["x0_x"]
    x0_m = lay["x0_m"]
    x0_y = lay["x0_y"]
    x0_w = lay["x0_w"]
    y_top = lay["y_top"]
    y0_w = lay["y0_w"]
    bw_x = lay["bw_x"]
    bh = lay["bh"]
    bw_y = lay["bw_y"]
    bh_y = lay["bh_y"]
    bw_w = lay["bw_w"]
    bh_w = lay["bh_w"]
    path_b_mid_x = lay["path_b_mid_x"]
    ox = lay["ox"]
    oy = lay["oy"]
    boxes = {
        "x": (x0_x, y_top, bw_x, bh, _capitalize_words("SN participation\ncoefficient (PC)")),
        "m": (x0_m, y_top, bw_x, bh, _capitalize_words("Global\nmodularity")),
        "y": (
            x0_y,
            y_top,
            bw_y,
            bh_y,
            _capitalize_words("Behavioral\ncoordination\n($RT_{\\mathrm{Diff}}$)"),
        ),
        "w": (x0_w, y0_w, bw_w, bh_w, "Adaptive Reserve\n(Susceptibility)"),
    }
    imm_y = y_top + bh + 0.65
    content_cx = lay["content_cx"]
    content_cy = lay["content_cy"]

    def sx(x: float) -> float:
        return x + ox

    def sy(y: float) -> float:
        return y + oy

    box_centers: dict[str, tuple[float, float]] = {}
    box_alpha = LAYOUT["d_box_alpha"]
    for key, (x0, y0, w, h, lab) in boxes.items():
        base_fc = CD_GREEN_LIGHT if key == "w" else "white"
        fc = mpl.colors.to_rgba(base_fc, alpha=box_alpha)
        ax.add_patch(
            mpatches.FancyBboxPatch(
                (sx(x0), sy(y0)),
                w,
                h,
                boxstyle="round,pad=0.02,rounding_size=0.08",
                fc=fc,
                ec=TEXT,
                lw=0.9,
                clip_on=False,
            )
        )
        box_centers[key] = (sx(x0 + w / 2), sy(y0 + h / 2))
        ax.text(
            sx(x0 + w / 2),
            sy(y0 + h / 2),
            lab,
            ha="center",
            va="center",
            fontsize=FS_D_DIAG_TICK,
            clip_on=False,
        )

    def arrow(x1, y1, x2, y2, color=TEXT, lw=1.2):
        ax.annotate(
            "",
            xy=(sx(x2), sy(y2)),
            xytext=(sx(x1), sy(y1)),
            arrowprops=dict(arrowstyle="-|>", color=color, lw=lw, shrinkA=0, shrinkB=0),
            clip_on=False,
        )

    y_mid = y_top + bh / 2
    beta_y = y_mid + LAYOUT["d_beta_y_offset"] - 0.08
    arr_pad = 0.14
    arrow(x0_x + bw_x + arr_pad, y_mid, x0_m - arr_pad, y_mid, CD_GREEN, 1.6)
    ax.text(
        sx((x0_x + bw_x + x0_m) / 2),
        sy(beta_y),
        f"$\\beta$ = {med['a1']:.3f}{_sig_stars(med['p_a1'])}",
        ha="center",
        fontsize=FS_D_DIAG_SMALL,
        color=CD_GREEN,
        clip_on=False,
    )
    arrow(x0_m + bw_x + arr_pad, y_mid, x0_y - arr_pad, y_mid, CD_GREEN, 1.6)
    ax.text(
        sx((x0_m + bw_x + x0_y) / 2),
        sy(beta_y),
        f"$\\beta$ = {med['b1']:.3f}{_sig_stars(med['p_b1'])}",
        ha="center",
        fontsize=FS_D_DIAG_SMALL,
        color=CD_GREEN,
        clip_on=False,
    )
    int_arrow_base = y0_w + bh_w + LAYOUT["d_int_arrow_gap"]
    arrow(path_b_mid_x, int_arrow_base, path_b_mid_x, y_mid, CD_GREEN, 1.6)
    int_label_y = (int_arrow_base + y_mid) / 2
    ax.text(
        sx(path_b_mid_x + LAYOUT["d_int_label_dx"]),
        sy(int_label_y),
        f"$\\beta_{{int}}$ = {med['b3']:.3f}{_sig_stars(med['p_b3'])}",
        ha="left",
        fontsize=FS_D_DIAG_SMALL,
        color=CD_GREEN,
        rotation=90,
        va="center",
        clip_on=False,
    )
    ax.text(
        sx(content_cx),
        sy(imm_y),
        (
            "Index of Moderated Mediation = "
            f"{med['imm']:.3f} [95% CI: {med['imm_ci'][0]:.3f}, {med['imm_ci'][1]:.3f}]"
        ),
        ha="center",
        va="bottom",
        fontsize=FS_D_SMALL,
        color="#333",
        clip_on=False,
    )
    path_a_mid_x = (x0_x + bw_x + x0_m) / 2
    fs_c_xlabel = FS  # same as panel C ax.set_xlabel (axes.labelsize)
    ci_a_lo, ci_a_hi = med["ci_a1"]
    ax.text(
        sx(path_a_mid_x),
        sy(content_cy + LAYOUT["d_path_a_stat_dy"]),
        (
            f"Path $a$: $\\beta$ = {med['a1']:.3f}{_sig_stars(med['p_a1'])} "
            f"[{ci_a_lo:.3f}, {ci_a_hi:.3f}]"
        ),
        ha="center",
        va="center",
        fontsize=fs_c_xlabel,
        color="#333",
        clip_on=False,
    )
    path_lbl_y = sy(y0_w + bh_w + LAYOUT["d_path_a_lbl_pad"])
    ax.text(
        sx(path_a_mid_x),
        path_lbl_y,
        "Hayes Model 14",
        ha="center",
        va="bottom",
        fontsize=fs_c_xlabel,
        color="#333",
        clip_on=False,
    )
    note_y = y0_w - 0.72
    ih_lo, ih_hi = med["ind_high_ci"]
    il_lo, il_hi = med["ind_low_ci"]
    note_x = sx(x0_y + bw_y)
    note_y_low = sy(note_y)
    note_line_dy = LAYOUT["d_note_line_dy"]
    ax.text(
        note_x,
        note_y_low,
        (
            f"Low Reserve ($W=+1$ $SD$): {med['ind_low']:.3f} "
            f"[95% CI: {il_lo:.3f}, {il_hi:.3f}]"
        ),
        ha="right",
        va="bottom",
        fontsize=fs_c_xlabel,
        color="#333",
        clip_on=False,
    )
    ax.text(
        note_x,
        note_y_low + note_line_dy,
        (
            f"High Reserve ($W=-1$ $SD$): {med['ind_high']:.3f} "
            f"[95% CI: {ih_lo:.3f}, {ih_hi:.3f}]"
        ),
        ha="right",
        va="bottom",
        fontsize=fs_c_xlabel,
        color="#333",
        clip_on=False,
    )


def _widen_axes(ax: plt.Axes, width_scale: float, dx: float = 0.0) -> None:
    pos = ax.get_position()
    extra = pos.width * (width_scale - 1.0)
    ax.set_position([pos.x0 + dx - extra * 0.35, pos.y0, pos.width * width_scale, pos.height])


